# <i>Random Forest</i>

### Imports

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

### Utility Functions

In [ ]:
def gini(y):
    classes, counts = np.unique(y, return_counts=True)
    prob = counts / counts.sum()
    return 1 - np.sum(prob ** 2)


def split_dataset(X, y, feature, threshold):
    left_mask = X[:, feature] <= threshold
    right_mask = ~left_mask
    return X[left_mask], y[left_mask], X[right_mask], y[right_mask]


### Best Split Finder

In [ ]:
def best_split(X, y, num_features):
    best_feature, best_threshold, best_gini = None, None, float("inf")

    feature_idxs = np.random.choice(X.shape[1], num_features, replace=False)

    for feature in feature_idxs:
        thresholds = np.unique(X[:, feature])
        for threshold in thresholds:
            X_l, y_l, X_r, y_r = split_dataset(X, y, feature, threshold)
            if len(y_l) == 0 or len(y_r) == 0:
                continue

            g = (len(y_l) * gini(y_l) + len(y_r) * gini(y_r)) / len(y)
            if g < best_gini:
                best_gini = g
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold


### decision Tree

In [ ]:
class TreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


class DecisionTree:
    def __init__(self, max_depth=10, min_samples=50, num_features=None):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.num_features = num_features
        self.root = None

    def build_tree(self, X, y, depth=0):
        if (len(y) < self.min_samples or
            depth == self.max_depth or
            len(np.unique(y)) == 1):
            return TreeNode(value=Counter(y).most_common(1)[0][0])

        feature, threshold = best_split(X, y, self.num_features)
        if feature is None:
            return TreeNode(value=Counter(y).most_common(1)[0][0])

        X_l, y_l, X_r, y_r = split_dataset(X, y, feature, threshold)
        left = self.build_tree(X_l, y_l, depth + 1)
        right = self.build_tree(X_r, y_r, depth + 1)

        return TreeNode(feature, threshold, left, right)

    def fit(self, X, y):
        self.root = self.build_tree(X, y)

    def predict_row(self, node, row):
        if node.value is not None:
            return node.value
        if row[node.feature] <= node.threshold:
            return self.predict_row(node.left, row)
        return self.predict_row(node.right, row)

    def predict(self, X):
        return np.array([self.predict_row(self.root, row) for row in X])


In [ ]:
class RandomForest:
    def __init__(self, n_trees=20, max_depth=10, min_samples=50):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.trees = []

    def bootstrap_sample(self, X, y):
        idxs = np.random.choice(len(X), len(X), replace=True)
        return X[idxs], y[idxs]

    def fit(self, X, y):
        self.trees = []
        num_features = int(np.sqrt(X.shape[1]))

        for _ in range(self.n_trees):
            X_sample, y_sample = self.bootstrap_sample(X, y)
            tree = DecisionTree(
                max_depth=self.max_depth,
                min_samples=self.min_samples,
                num_features=num_features
            )
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        return np.array([
            Counter(tree_preds[:, i]).most_common(1)[0][0]
            for i in range(X.shape[0])
        ])


In [ ]:
def train_test_split_manual(X, y, test_size=0.2):
    idx = np.random.permutation(len(X))
    test_len = int(len(X) * test_size)

    test_idx = idx[:test_len]
    train_idx = idx[test_len:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


In [ ]:
X = df.drop('cardio', axis=1).values
y = df['cardio'].values

X_train, X_test, y_train, y_test = train_test_split_manual(X, y)

rf = RandomForest(
    n_trees=25,
    max_depth=12,
    min_samples=50
)

rf.fit(X_train, y_train)

y_train_pred = rf.predict(X_train)
y_test_pred = rf.predict(X_test)


In [ ]:
train_acc = np.mean(y_train_pred == y_train)
test_acc = np.mean(y_test_pred == y_test)

print("Train Accuracy:", round(train_acc, 4))
print("Test Accuracy :", round(test_acc, 4))
print("Gap           :", round(train_acc - test_acc, 4))
